# Multiclass glass classification

In this practical, we will look at multiclass classification - predicting one of many classes, rather than just one of two.

We'll be working with data from this 1987 forensic science paper: [Rule Induction in Forensic Science](http://www0.cs.ucl.ac.uk/staff/W.Langdon/ftp/papers/evett_1987_rifs.pdf)

In this paper, the authors classified fragments of glass into one of six classes, based on their chemical composition. This was motivated by the need to determine whether fragments of glass found on a suspect's clothing was of the same type of that at the crime scene.

The training data consists of 214 items. Each item has ten features and a type - the features are [Refractive Index](https://en.wikipedia.org/wiki/Refractive_index) and the weight percent in oxide form of 8 elements.

| Feature 	|    Description    	|
|:-------:	|:-----------------:	|
|    RI   	| Refractive Index  	|
|    Na   	| Sodium            	|
|    Mg   	| Magnesium          	|
|    Al   	| Aluminum          	|
|    Si   	| Silicon           	|
|    K    	| Potassium          	|
|    Ca   	| Calcium           	|
|    Ba   	| Barium            	|
|    Fe   	| Iron              	|

There are six types of glass: containers, tablewear, headlamps, vehicle, [float-processed](https://en.wikipedia.org/wiki/Float_glass) and non-float-processed building glass.


# Data exploration

First, let's look at the data. Below, after loading it in, look at the class balance with the `.value_counts()` applied to the relevant column. What do you observe?

In [ ]:
import pandas as pd

data = pd.read_csv('data/glass.csv')

# Your code below



For each class, what is the the average value and standard deviation per variable? What do you observe?

Which features do you think will be most useful for predicting which classes?

Will you need to do any data pre-processing?

(Hint: use `.groupby()` on a dataframe and then apply `.agg()`. This can take a list of arguments you want aggregate data for, such as `mean` and `std`.)

In [ ]:
# Your code below



# Standardisation

As the features do not have the same scale, let's fix that so that they have mean 0 and stdev 1 and it will be easier for us to examine and also easier for models to learn from.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Rescale the features to have std 1 and mean 0
scaler = StandardScaler()
columns = ['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe']
data[columns] = scaler.fit_transform(data[columns])

The cell below will show the distribution of the feature values for each class.

Which classes, if any, have a distinctive distribution?

In [ ]:
import seaborn as sns

# Grid lines make it easier to compare along rows
sns.set_style('whitegrid') 

import matplotlib.pyplot as plt

# Set up the size of the plot
plt.figure(figsize=(20,8)) 

# Long-format data is easier to plot with seaborn
long_data = data.melt(id_vars=['Type'], var_name='Feature')

# Box plot
sns.boxplot(data=long_data, x='Type', hue='Feature', y='value')

# Tidy up layout
plt.tight_layout()


#Your thoughts below


# Setting up data

The normalised data is still in a DataFrame. Split it into two parts - the features `X` and the class labels `y`

In [ ]:
#Your code below


Now, create train/test sets: `X_train`, `X_test`, `y_train` and `y_test`.

As we have only around 200 data points, use 20% of data for testing rather than the default 25%. Use `random_state=5` to get the same results each time.

(You can use the `test_size` parameter for this.)

In [ ]:
from sklearn.model_selection import train_test_split

#Your code below


# Training and evaluating models

The original paper from 1987 used [BEAGLE](https://www.emerald.com/insight/content/doi/10.1108/eb005587/full/html) to classify their data - a rule-based algorithm from 1981 for classifying data, inspired by Darwinian evolution. Rules like "were evaluated for their "goodness" and only the best rules kept.

Let's compare some models which are, by default, multiclass to some models which need to use a multiclass strategy such as 'One-V-One' or 'One-V-Rest'.

After they have been imported below:

* instantiate one of each model. (Use the default hyperparameters, except for the SVM - set `max_iter=2000` otherwise the model will not converge. And for LR/SVM set `random_state=5`)
* use the `.fit()` method of each to train the model using the training data (`X_train`, `y_train`)
* then use each model's `.score()` method on the training data (to see if they overfitted)
* and then again on the test data (to see if they generalised)

In [ ]:
# Multiclass by default
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Binary only
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

#Your code below


The DT overfits, as usual, but generalises well.

The other models perform around the same on unseen data, though KNN (the simplest model) is the worst overall.

The SVM actually does slightly better on the unseen data! But this is likely due to randomisation/initialisation issues.

# Looking at per-class performance

The accuracy score gives one impression of how well a model does, but the per-class peformance gives a much more nuanced picture.

Use the `classification_report` function to see how well each model performed on each of the six classes.

To do this, use each trained models `.predict()` function on the `X_test` data. Then pass these predictions and the true labels `y_test` to `classification_report`.

Store the predictions of each model as `dt_pred` `kn_pred` `svm_pred` `lm_pred`

What do you notice?

In [ ]:
from sklearn.metrics import classification_report

#Your code and thoughts below


# Looking at per-class performance

Another way to look at per-class performance is through a confusion matrix.

This shows which items got classified as what and allows you to see patterns, such as whether one class is always predicted as another.

The `confusion_matrix` function takes two lists: the true labels and the predicted ones.

It returns a matrix where rows are the true labels, columns are the predicted.

A cell at $(i,j)$ denotes the number of items of true class $i$ that were labelled with $j$.

Ideally you want all cells to have a value of 0 except on the diagonal.

The order of the rows (top to bottom) and columns (left to right) will be the same and will be in alphabetical order by default.

In [ ]:
from sklearn.metrics import confusion_matrix

for predictions, model in zip([dt_pred, kn_pred, svm_pred, lr_pred], ['DT', 'KNN', 'SVM', 'LR']):
    print(f'Confusion matrix for {model}:')
    print(confusion_matrix(predictions, y_test))
    print()

# Looking at per-class performance

It's easier to understand the confusion matrix when plotted as a heatmap. `seaborn` has a useful function for this.

Run the cell below. What do you observe?

In [ ]:
f, axs = plt.subplots(2,2, figsize=(10,10))

axs = axs.flatten()
labels = list(y_test.unique())

for preds, model, ax in zip([dt_pred, kn_pred, svm_pred, lr_pred], ['DT', 'KNN', 'SVM', 'LR'], axs):
    matrix = confusion_matrix(preds, y_test)
    sns.heatmap(matrix, square=True, annot=False, cmap='Reds', ax=ax, cbar=False)
    ax.set(title=model)
    ax.set(title=model, xticklabels='', yticklabels=labels)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()

#Your thoughts below


# OVR vs OVO

The sklearn models use OVR for their multiclass strategy. Would using OVO help?

Using the `OneVsOneClassifier` class from `sklearn.multiclass`, fit and train and evaluate it just like you would any other sklearn model.

How does it compare to the OVR strategy when you compare them by `classification_report`?


In [ ]:
from sklearn.multiclass import OneVsOneClassifier

ovo = OneVsOneClassifier(estimator=LinearSVC(max_iter=2000, dual="auto"))

#Your code and thoughts below


# Conclusion

Working with multiclass data doesn't generally require the use of a specific model - the binary classifiers can be adapted and still perform very well.

For small datasets it may still be worth evaluating different strategies. As we saw here, it helped a little. But you would likely get more of an improvement from better finetuning of model hyperparameters through grid search and cross-validation. Ideally, you would get more data!